In [1]:
# Module 3 — CNN (Vehicle Damage Image Classification) 
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Define paths
DATASET_DIR = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\data\raw\car_damage\data1a"
BASE_DIR = r"C:\Users\Prasanth Rajaram\InsureAI_Local"
MODEL_DIR = os.path.join(BASE_DIR, "models")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

def load_dataset(dataset_dir, target_size=(128, 128)):
    """
    Loads images from dataset directory (training/validation subdirectories)
    and returns numpy arrays for images and labels.
    """
    class_map = {"00-damage": 0, "01-whole": 1}
    X, y = [], []
    
    for class_name, class_idx in class_map.items():
        class_path = os.path.join(dataset_dir, class_name)
        if not os.path.exists(class_path):
            continue
            
        for file_name in os.listdir(class_path):
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_path, file_name)
                try:
                    # Load, convert to RGB, and resize
                    img = Image.open(img_path).convert("RGB")
                    img_resized = img.resize(target_size, Image.Resampling.BILINEAR)
                    img_arr = np.array(img_resized, dtype=np.float32) / 255.0 # Normalize pixel values to [0, 1]
                    X.append(img_arr)
                    y.append(class_idx)
                except Exception as e:
                    print(f"Warning: Failed to load image {img_path}: {e}")
                    
    return np.array(X), np.array(y)

# ==========================================
# 1. APPROACH 1: CUSTOM CNN FROM SCRATCH
# ==========================================
def build_scratch_cnn(input_shape=(128, 128, 3)):
    """
    Designs a simple CNN from scratch.
    Conv2D -> MaxPooling -> Flatten -> Dense layers.
    """
    model = models.Sequential(name="Scratch_Damage_CNN")
    
    # Input Layer
    model.add(layers.Input(shape=input_shape))
    
    # Convolution Block 1
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1'))
    model.add(layers.BatchNormalization(name='bn1'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool1'))
    model.add(layers.Dropout(0.25, name='dropout1'))
    
    # Convolution Block 2
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2'))
    model.add(layers.BatchNormalization(name='bn2'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool2'))
    model.add(layers.Dropout(0.25, name='dropout2'))
    
    # Dense Classification Head
    model.add(layers.Flatten(name='flatten'))
    model.add(layers.Dense(128, activation='relu', name='dense1'))
    model.add(layers.BatchNormalization(name='bn_dense1'))
    model.add(layers.Dropout(0.5, name='dropout_dense1'))
    
    # Output layer (2 classes: damaged, whole)
    model.add(layers.Dense(2, activation='softmax', name='output'))
    
    return model

# ==========================================
# 2. APPROACH 2: TRANSFER LEARNING (MobileNetV2)
# ==========================================
def build_transfer_learning_model(input_shape=(128, 128, 3)):
    """
    Loads pre-trained MobileNetV2 model, freezes base layers, and adds custom classification head.
    """
    inputs = layers.Input(shape=input_shape, name="input_layer")
    
    # Scale inputs manually from [0, 1] to [-1, 1] to avoid preprocessing scale bugs
    x_preprocessed = inputs * 2.0 - 1.0
    
    # Load base model with ImageNet weights, excluding top classification layers
    base_model = tf.keras.applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    # Freeze pre-trained weights
    base_model.trainable = False
    
    # Connect pipeline
    x = base_model(x_preprocessed, training=False)
    x = layers.GlobalAveragePooling2D(name="global_pooling")(x)
    x = layers.Dense(128, activation='relu', name='dense1')(x)
    x = layers.BatchNormalization(name='bn_dense1')(x)
    x = layers.Dropout(0.5, name='dropout_dense1')(x)
    outputs = layers.Dense(2, activation='softmax', name='output_layer')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Transfer_MobileNetV2")
    return model, base_model

def plot_and_save_history(history, model_name, plot_save_path):
    """
    Plots training vs validation accuracy and loss curves.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1.plot(history.history['loss'], label='Train Loss', color='#3f72af', linestyle='--')
    ax1.plot(history.history['val_loss'], label='Val Loss', color='#3f72af')
    ax1.set_title(f"{model_name} - Loss Curves", fontweight='bold')
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)
    
    # Accuracy curves
    ax2.plot(history.history['accuracy'], label='Train Accuracy', color='#ff7e67', linestyle='--')
    ax2.plot(history.history['val_accuracy'], label='Val Accuracy', color='#ff7e67')
    ax2.set_title(f"{model_name} - Accuracy Curves", fontweight='bold')
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(plot_save_path, dpi=150)
    plt.close()

def main():
    print("=" * 80)
    print("MODULE 3: VEHICLE DAMAGE IMAGE CLASSIFICATION PIPELINE")
    print("=" * 80)
    
    # 1. Load Data
    train_dir = os.path.join(DATASET_DIR, "training")
    val_dir = os.path.join(DATASET_DIR, "validation")
    
    X_train, y_train = load_dataset(train_dir)
    X_val, y_val = load_dataset(val_dir)
    
    print(f"\nDataset loaded:")
    print(f"  - Training Set: {X_train.shape[0]} images (damaged: {sum(y_train==0)}, whole: {sum(y_train==1)})")
    print(f"  - Validation Set: {X_val.shape[0]} images (damaged: {sum(y_val==0)}, whole: {sum(y_val==1)})")
    
    # 2. Data Augmentation
    # Set up generator with rotation, zoom, and flips for training only
    train_datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    # Training parameters
    batch_size = 32
    epochs = 20
    
    # ==========================================
    # A. TRAIN CUSTOM CNN FROM SCRATCH
    # ==========================================
    print("\n" + "-" * 50)
    print("TRAINING CUSTOM CNN MODEL FROM SCRATCH")
    print("-" * 50)
    
    scratch_model = build_scratch_cnn()
    scratch_model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    early_stopping_scratch = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
    
    history_scratch = scratch_model.fit(
        train_datagen.flow(X_train, y_train, batch_size=batch_size),
        validation_data=(X_val, y_val),
        epochs=epochs,
        callbacks=[early_stopping_scratch],
        verbose=1
    )
    
    plot_scratch_path = os.path.join(REPORTS_DIR, "scratch_cnn_training_curves.png")
    plot_and_save_history(history_scratch, "Scratch CNN", plot_scratch_path)
    print(f"Saved Scratch CNN training curves to {plot_scratch_path}")
    
    # Evaluate Custom CNN
    scratch_preds_prob = scratch_model.predict(X_val)
    scratch_preds = np.argmax(scratch_preds_prob, axis=1)
    
    scratch_acc = accuracy_score(y_val, scratch_preds)
    scratch_prec = precision_score(y_val, scratch_preds, zero_division=0)
    scratch_rec = recall_score(y_val, scratch_preds, zero_division=0)
    scratch_f1 = f1_score(y_val, scratch_preds, zero_division=0)
    scratch_cm = confusion_matrix(y_val, scratch_preds)
    
    # ==========================================
    # B. TRAIN TRANSFER LEARNING MODEL (MobileNetV2)
    # ==========================================
    print("\n" + "-" * 50)
    print("TRAINING TRANSFER LEARNING MODEL (MobileNetV2)")
    print("-" * 50)
    
    tl_model, base_model = build_transfer_learning_model()
    tl_model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    early_stopping_tl = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
    
    history_tl = tl_model.fit(
        train_datagen.flow(X_train, y_train, batch_size=batch_size),
        validation_data=(X_val, y_val),
        epochs=epochs,
        callbacks=[early_stopping_tl],
        verbose=1
    )
    
    plot_tl_path = os.path.join(REPORTS_DIR, "transfer_learning_training_curves.png")
    plot_and_save_history(history_tl, "Transfer Learning MobileNetV2", plot_tl_path)
    print(f"Saved Transfer Learning training curves to {plot_tl_path}")
    
    # Evaluate Transfer Learning
    tl_preds_prob = tl_model.predict(X_val)
    tl_preds = np.argmax(tl_preds_prob, axis=1)
    
    tl_acc = accuracy_score(y_val, tl_preds)
    tl_prec = precision_score(y_val, tl_preds, zero_division=0)
    tl_rec = recall_score(y_val, tl_preds, zero_division=0)
    tl_f1 = f1_score(y_val, tl_preds, zero_division=0)
    tl_cm = confusion_matrix(y_val, tl_preds)
    
    # ==========================================
    # C. COMPARE MODELS AND SAVE BEST
    # ==========================================
    print("\n" + "=" * 80)
    print("MODEL COMPARISON AND EVALUATION")
    print("=" * 80)
    
    print("\n--- PERFORMANCE SUMMARY ---")
    print(f"{'Metric':<18} | {'Scratch CNN':<15} | {'Transfer Learning':<20}")
    print("-" * 60)
    print(f"{'Val Accuracy':<18} | {scratch_acc:<15.4f} | {tl_acc:<20.4f}")
    print(f"{'Val Precision':<18} | {scratch_prec:<15.4f} | {tl_prec:<20.4f}")
    print(f"{'Val Recall':<18} | {scratch_rec:<15.4f} | {tl_rec:<20.4f}")
    print(f"{'Val F1-Score':<18} | {scratch_f1:<15.4f} | {tl_f1:<20.4f}")
    print("-" * 60)
    
    print("\n--- CONFUSION MATRICES ---")
    print("Scratch CNN Confusion Matrix:")
    print(scratch_cm)
    print("\nTransfer Learning (MobileNetV2) Confusion Matrix:")
    print(tl_cm)
    print("=" * 80 + "\n")
    
    # Determine the best model
    best_model_name = ""
    best_model = None
    
    if tl_acc >= scratch_acc:
        best_model_name = "Transfer Learning (MobileNetV2)"
        best_model = tl_model
    else:
        best_model_name = "Scratch CNN"
        best_model = scratch_model
        
    print(f"--> Best model determined: {best_model_name} with Accuracy {max(scratch_acc, tl_acc):.4f}")
    
    # Save the best model
    best_model_save_path = os.path.join(MODEL_DIR, "best_vehicle_damage_model.keras")
    best_model.save(best_model_save_path)
    print(f"Saved the best model to {best_model_save_path}\n")

if __name__ == "__main__":
    main()


MODULE 3: VEHICLE DAMAGE IMAGE CLASSIFICATION PIPELINE

Dataset loaded:
  - Training Set: 1840 images (damaged: 920, whole: 920)
  - Validation Set: 460 images (damaged: 230, whole: 230)

--------------------------------------------------
TRAINING CUSTOM CNN MODEL FROM SCRATCH
--------------------------------------------------
Epoch 1/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 40s 643ms/step - accuracy: 0.5788 - loss: 1.0204 - val_accuracy: 0.5087 - val_loss: 1.1943
Epoch 2/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 40s 682ms/step - accuracy: 0.6217 - loss: 0.7919 - val_accuracy: 0.5000 - val_loss: 2.9462
Epoch 3/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 42s 722ms/step - accuracy: 0.6207 - loss: 0.7459 - val_accuracy: 0.5500 - val_loss: 0.9970
Epoch 4/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 50s 859ms/step - accuracy: 0.6473 - loss: 0.7080 - val_accuracy: 0.6022 - val_loss: 0.7413
Epoch 5/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 46s 789ms/step - accuracy: 0.6533 - loss: 0.6534 - val_accuracy: 0.6652 - val_loss: 0.6136
Epoch 6/20
58/58 ━━━━━━━━━